In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from pathlib import Path
import json
from cleaning.profile import profile
import cleaning.checks as chk 
from dataclasses import asdict   # only if the script itself converts results to a table



RAW = Path('C:/repositaries/4.personal/EEA_interview/data/raw')

source_code= 'env_wasmun'
key_dimension = 'wst_oper'

#----
file_path = Path(RAW, f'{source_code}')


#read the parquet file
df = pd.read_parquet(file_path/'df.parquet')
lookups = json.load(open(file_path / 'lookup.json', encoding='utf-8'))



In [20]:
#check that the time frequency of the dataset is annually only
annual_only = (df['freq'] == "A").all()
print(annual_only)
if annual_only:
    df['time'] = df['time'].astype(int)


print(df.dtypes)




True
freq            str
wst_oper        str
unit            str
geo             str
time          int64
value       float64
dtype: object


In [ ]:
profile(df,lookups, key_dimension)
#If time frequency is only A, transform time to integer
# using the lookups of countrycodes keep only EU27
#drop variable freq
#unit to int

shape: (11100, 6)

dtypes:
freq            str
wst_oper        str
unit            str
geo             str
time            str
value       float64

Time frequency:
A: 11100

missing share per column:
freq        0.000
wst_oper    0.000
unit        0.000
geo         0.000
time        0.000
value       0.154

categories with negative values
All values are positive

constant columns: ['freq']

value summary:
count     9394.000000
mean      1584.664254
std       5086.150290
min          0.000000
25%         20.000000
50%        217.000000
75%        692.000000
max      53871.000000

lookup vs data (code counts/values in dimension):
  freq: lookup=1, in data=1
  wst_oper: lookup=10, in data=10
  unit: lookup=2, in data=2
  geo: lookup=37, in data=37
  time: lookup=15, in data=15


In [ ]:
#create a function that checks for sampling completion: which countries didnot report which indicator in which year
# cols: country, year, indicator_missing

results = [
    chk.unique_key(df, key=[key_dimension, "geo", "time"]),
    chk.codes_in_lookup(df, lookups),
    chk.non_negative(df),
    chk.numeric_dtypes(df),
    chk.missing_share(df, max_share=0.0, severity="warning"),
    chk.missing_share(df,  max_share=0.5, severity="error"),
    chk.constant_columns(df),
    chk.dims_match_lookup(df, lookups),
]

report = pd.DataFrame(asdict(r) for r in results)

report

,check,severity,passed,n_issues,detail
0,unique_key,error,False,11100,"key=['wst_oper', 'geo', 'time']"
1,codes_in_lookup,error,True,0,{}
2,non_negative,error,True,0,value
3,numeric_dtypes,warning,False,1,"{'time': 'dtype=str, not convertible=0'}"
4,missing_share_2,warning,False,1,['value: 15.37%']
5,missing_share_2,error,True,0,[]
6,constant_columns,info,False,1,['freq']
7,dims_match_lookup,error,True,0,"n_lookup=5, n_data=5; only in lookup=[]; only ..."


In [6]:
lookups[key_dimension]

{'GEN': 'Waste generated',
 'TRT': 'Waste treatment',
 'DSP_I_RCV_E': 'Disposal - incineration (D10) and recovery - energy recovery (R1)',
 'DSP_L_OTH': 'Disposal - landfill and other (D1-D7, D12)',
 'DSP_I': 'Disposal - incineration (D10)',
 'RCV_E': 'Recovery - energy recovery (R1)',
 'RCY': 'Recycling',
 'RCY_M': 'Recycling - material',
 'RCY_C_D': 'Recycling - composting and digestion',
 'PRP_REU': 'Preparing for reuse'}